# 1. 최근 간격을 알면 다음 간격을 더 잘 설명할까?
B2는 **직전 행동**만 봅니다. B3는 **그 전에 끝난 입력 간격 하나**도 봅니다.
예: 왼쪽 →(80ms) 회전 →(?) 다음 행동. 회전 시점에는 80ms가 이미 관측됐지만 ?는 아직 모릅니다.

이번에는 기존 Train/Validation만 탐색합니다. 이미 평가한 Test는 열지 않으며 새 후보의 최종 검증에는 별도 미관측 cohort가 필요합니다.
기록된 처리 시도 간격이지 실제 키 누름·반응시간이 아닙니다. Hold·배치·다음 행동·신경망을 학습하지 않습니다.


In [ ]:
import hashlib
import json
import math
import os
import platform
from collections import Counter
from pathlib import Path

import numpy as np

EXPERIMENT_ID = "human-timing-history-baseline"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_history_baseline.ipynb"
BASELINE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_baseline.ipynb"
BASELINE_DEFS_SHA = "f1943f93868c31d03ff942b90a2a3effbd083f8b8801e3729848916bfba467fb"
selected_ids = {"shared-definitions", "input-definitions", "metric-definitions", "model-definitions"}
definition_cells = [{"id": c["id"], "source": "".join(c["source"])}
                    for c in json.loads(BASELINE_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
                    if c["id"] in selected_ids]
definition_bytes = (json.dumps(definition_cells, ensure_ascii=False, sort_keys=True,
                              allow_nan=False, indent=2) + "\n").encode("utf-8")
if len(definition_cells) != 4 or hashlib.sha256(definition_bytes).hexdigest() != BASELINE_DEFS_SHA:
    raise ValueError("BASELINE_DEFINITIONS_CHANGED")
SHARED_NOTEBOOK_SHA = "7ccaebcb5db7c1e5c15067f3bceebc793b3ded2c81ccb414834b31d0959ad8ed"
shared_notebook = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
if hashlib.sha256(shared_notebook.read_bytes()).hexdigest() != SHARED_NOTEBOOK_SHA:
    raise ValueError("SHARED_DEPENDENCY_CHANGED")
baseline = {"__name__": "frozen_b2_shared"}
for cell in definition_cells:
    exec(compile(cell["source"], str(BASELINE_NOTEBOOK), "exec"), baseline)
baseline["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "finite", "integer", "json_bytes", "sha256_file", "iter_rows",
             "reject_constant", "checked_paths", "publish_report", "validate_sample",
             "prepare_distribution", "crps", "distribution_summary", "forecast_values", "load_inputs", "ACTIONS"):
    globals()[name] = baseline[name]


from bisect import bisect_right
from collections import defaultdict


# 2. 이력 연결과 작은 예제
같은 game/player/piece/sourceLine 안에서 index가 정확히 이어질 때만 직전 간격을 연결합니다.
첫 pair와 빠진 pair 뒤에는 이력이 없습니다. 다른 조각·사람·구간을 건너뛰어 연결하지 않습니다.

Train의 과거 간격에서 1/3·2/3 nearest-rank 경계를 정합니다. 예를 들어 경계가 [80, 160]이면
80 미만 / 80 이상 160 미만 / 160 이상입니다. 같은 경계는 그대로 유지합니다.
B3는 행동×간격 구간의 경험 분포를 쓰고, 이력이 없거나 학습 cell이 비었으면 B2로 돌아갑니다.
Validation에서는 이전 실측 간격을 관측한 one-step 예측만 검사합니다. 자기 예측으로 이어가는 실행 검사는 아닙니다.


In [ ]:
def with_history(rows):
    """동일 원본 입력 구간의 인접 pair만 연결하고 모든 현재 표본을 보존한다."""
    require(rows, "EMPTY_ROWS")
    indexed, ids = {}, set()
    for row in rows:
        validate_sample(row, row["split"], {row["gameId"]})
        require(row["split"] in ("train", "validation"), "TEST_NOT_ALLOWED")
        key = tuple(row[k] for k in ("split", "gameId", "playerNumber", "pieceIndex", "sourceLine"))
        key += (row["currentInputIndex"],)
        require(key not in indexed and row["sampleId"] not in ids, "DUPLICATE_LINEAGE")
        indexed[key] = row
        ids.add(row["sampleId"])
    result = []
    for row in sorted(rows, key=lambda r: r["sampleId"]):
        key = tuple(row[k] for k in ("split", "gameId", "playerNumber", "pieceIndex", "sourceLine"))
        prior = indexed.get(key + (row["previousInputIndex"],))
        if prior is not None:
            require(prior["nextActionDiagnostic"] == row["previousAction"], "HISTORY_ACTION_MISMATCH")
        result.append({**row, "priorIntervalMs": None if prior is None else prior["recordedIntervalMs"],
                       "historyStatus": "available" if prior is not None else
                       ("segment_start" if row["previousInputIndex"] == 0 else "gap_or_missing")})
    return result


def fit_history(train):
    """Train만으로 두 경험 분포와 B3 구간 경계를 만든다."""
    require(train and all(r["split"] == "train" for r in train), "TRAIN_ONLY_REQUIRED")
    history = with_history(train)
    past = sorted(r["priorIntervalMs"] for r in history if r["priorIntervalMs"] is not None)
    require(past, "NO_TRAIN_HISTORY")
    cuts = [float(past[math.ceil(q * len(past)) - 1]) for q in (1 / 3, 2 / 3)]
    actions, cells = defaultdict(list), defaultdict(list)
    for row in history:
        actions[row["previousAction"]].append(row["recordedIntervalMs"])
        if row["priorIntervalMs"] is not None:
            key = row["previousAction"] + ":" + str(bisect_right(cuts, row["priorIntervalMs"]))
            cells[key].append(row["recordedIntervalMs"])
    return {"bundleSchemaVersion": 1, "features": ["previousAction", "priorIntervalMs"],
            "label": "recordedIntervalMs", "trainGameIds": sorted({r["gameId"] for r in train}),
            "trainPairs": len(train), "bucketCutsMs": cuts,
            "global": sorted(r["recordedIntervalMs"] for r in train),
            "actions": {k: sorted(v) for k, v in sorted(actions.items())},
            "historyCells": {k: sorted(v) for k, v in sorted(cells.items())},
            "canonicalArtifact": False, "productionIntegration": False}


def support_for(bundle, row, model):
    """예측 시점에 관측된 action과 과거 간격만 참조한다."""
    require(model in ("B2", "B3"), "MODEL_INVALID")
    if model == "B3":
        if row["priorIntervalMs"] is None:
            reason = "missing_history"
        else:
            key = row["previousAction"] + ":" + str(bisect_right(bundle["bucketCutsMs"], row["priorIntervalMs"]))
            if key in bundle["historyCells"]:
                return bundle["historyCells"][key], "history_cell"
            reason = "unseen_history_cell"
    else:
        reason = "action"
    if row["previousAction"] not in bundle["actions"]:
        return bundle["global"], "unseen_action_global"
    return bundle["actions"][row["previousAction"]], reason


In [ ]:
def evaluate_history(bundle, rows):
    """동일 Validation 행에서 두 모델을 평가하며 bundle을 수정하지 않는다."""
    require(rows and all(r["split"] == "validation" for r in rows), "VALIDATION_ONLY_REQUIRED")
    require(not set(bundle["trainGameIds"]) & {r["gameId"] for r in rows}, "GAME_SPLIT_LEAKAGE")
    before = json_bytes(bundle)
    history = with_history(rows)
    games = sorted({r["gameId"] for r in history})
    results, per_game, per_action, group_metrics = [], [], [], []
    for model in ("B2", "B3"):
        groups = defaultdict(list)
        for i, row in enumerate(history):
            support, reason = support_for(bundle, row, model)
            group = (row["previousAction"], reason,
                     bisect_right(bundle["bucketCutsMs"], row["priorIntervalMs"])
                     if model == "B3" and reason == "history_cell" else -1)
            groups[group].append(i)
        scores, absolute = np.empty(len(rows)), np.empty(len(rows))
        values, weights, fallbacks = [], [], Counter()
        for group, indices in sorted(groups.items()):
            support, reason = support_for(bundle, history[indices[0]], model)
            prepared = prepare_distribution(support)
            labels = [history[i]["recordedIntervalMs"] for i in indices]
            scores[indices] = crps(prepared, labels)
            median = distribution_summary(support)["medianMs"]
            absolute[indices] = np.abs(np.asarray(labels) - median)
            values.extend(support)
            weights.extend([len(indices) / len(support)] * len(support))
            fallbacks[reason] += len(indices)
        game_values = []
        for game in games:
            indices = [i for i, row in enumerate(history) if row["gameId"] == game]
            value = float(np.mean(scores[indices]))
            game_values.append(value)
            per_game.append({"model": model, "gameId": game, "pairs": len(indices), "crpsMs": value})
        for action in sorted({r["previousAction"] for r in history}):
            indices = [i for i, row in enumerate(history) if row["previousAction"] == action]
            per_action.append({"model": model, "action": action, "pairs": len(indices),
                               "crpsMs": float(np.mean(scores[indices]))})
        for available in (False, True):
            indices = [i for i, row in enumerate(history)
                       if (row["priorIntervalMs"] is not None) == available]
            group_metrics.append({"model": model, "historyAvailable": available, "pairs": len(indices),
                                  "pairMicroCrpsMs": float(np.mean(scores[indices])) if indices else None})
        x, inverse = np.unique(values, return_inverse=True)
        probability = np.minimum(np.cumsum(np.bincount(inverse, weights=weights)) / sum(weights), 1)
        probability[-1] = 1
        results.append({"model": model, "gameMacroCrpsMs": math.fsum(game_values) / len(games),
                        "pairMicroCrpsMs": float(np.mean(scores)), "medianMaeMs": float(np.mean(absolute)),
                        "routes": dict(sorted(fallbacks.items())), "predicted": distribution_summary(values, weights),
                        "predictedCdf": {"ms": x.tolist(), "probability": probability.tolist()}})
    by_game = {(g["model"], g["gameId"]): g["crpsMs"] for g in per_game}
    deltas = [{"gameId": g, "deltaB3MinusB2Ms": by_game["B3", g] - by_game["B2", g]} for g in games]
    labels = [r["recordedIntervalMs"] for r in history]
    x, counts = np.unique(labels, return_counts=True)
    delta = results[1]["gameMacroCrpsMs"] - results[0]["gameMacroCrpsMs"]
    require(json_bytes(bundle) == before, "BUNDLE_MUTATED")
    return {"games": len(games), "pairs": len(rows), "metrics": results, "perGame": per_game,
            "perAction": per_action, "historySubsets": group_metrics,
            "historyCounts": dict(sorted(Counter(r["historyStatus"] for r in history).items())),
            "observed": distribution_summary(labels),
            "observedCdf": {"ms": x.tolist(), "probability": (np.cumsum(counts) / len(labels)).tolist()},
            "gameComparison": deltas, "deltaB3MinusB2Ms": delta,
            "improvedGames": sum(g["deltaB3MinusB2Ms"] < -1e-12 for g in deltas),
            "tiedGames": sum(abs(g["deltaB3MinusB2Ms"]) <= 1e-12 for g in deltas),
            "worsenedGames": sum(g["deltaB3MinusB2Ms"] > 1e-12 for g in deltas),
            "evaluationFitCount": 0}


def synthetic_rows():
    """손으로 확인할 두 행동 구간. 실제 로그를 읽거나 생성기를 실행하지 않는다."""
    def segment(game, split, values):
        rows = []
        for i, value in enumerate(values):
            lineage = {"gameId": game, "playerNumber": 1, "pieceIndex": 1, "sourceLine": 1,
                       "previousInputIndex": i, "currentInputIndex": i + 1}
            rows.append({"schemaVersion": 1, "sampleId": hashlib.sha256(json_bytes(lineage)).hexdigest(),
                         **lineage, "previousAction": "move_left", "nextActionDiagnostic": "move_left",
                         "recordedIntervalMs": value, "split": split})
        return rows
    return {"train": segment("fixture-train", "train", [0, 50, 100, 150, 200, 250]),
            "validation": segment("fixture-validation", "validation", [0, 50, 100, 150])}


In [ ]:
def output_path(name, mode):
    """기존 공통 출력 보호를 적용하고 새 ignored report만 허용한다."""
    evidence = ROOT / "data/ml-experiments" / EXPERIMENT_ID
    evidence.mkdir(parents=True, exist_ok=True)
    output = Path(name)
    if not output.is_absolute():
        output = ROOT / output
    return checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(output), mode)[1]


def run_history(mode="smoke", output=None, synthetic=False):
    """입력 검증 → Train fit → Validation 평가 → 불변성 검사 → 선택적 독점 게시."""
    require(mode in ("smoke", "full") and not (mode == "full" and synthetic), "INVALID_RUN_MODE")
    require(mode != "full" or output is not None, "FULL_OUTPUT_REQUIRED")
    destination = output_path(output, mode) if output is not None else None
    watched = {SOURCE_NOTEBOOK: sha256_file(SOURCE_NOTEBOOK),
               BASELINE_NOTEBOOK: sha256_file(BASELINE_NOTEBOOK), shared_notebook: SHARED_NOTEBOOK_SHA}
    if synthetic:
        rows = synthetic_rows()
        provenance = {"kind": "synthetic_only"}
    else:
        manifest, rows, sources = load_inputs(
            os.environ.get("HUMAN_TIMING_HISTORY_DATASET", baseline["DEFAULT_DATASET"]), mode)
        watched.update(sources)
        provenance = {"kind": "fixed_legacy_train_validation", "manifestSha256": baseline["EXPECTED_MANIFEST_SHA"],
                      "files": {s: manifest["files"][s + ".jsonl"] for s in ("train", "validation")}}
    require(not {r["gameId"] for r in rows["train"]} & {r["gameId"] for r in rows["validation"]},
            "GAME_SPLIT_LEAKAGE")
    bundle = fit_history(rows["train"])
    result = evaluate_history(bundle, rows["validation"])
    train_history = with_history(rows["train"])
    report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": mode,
              "status": "timing_history_smoke_valid" if mode == "smoke" else "timing_history_validation_complete",
              "synthetic": synthetic, "fullCohortCovered": mode == "full",
              "policyDecisionEligible": mode == "full", "canonicalResultEligible": False,
              "productionIntegration": False, "testFileRead": False, "testEvaluated": False,
              "previousTestAlreadyObserved": True, "futureIndependentCohortRequired": True,
              "modelFitCount": 2, "evaluatedModels": ["B2", "B3"], "gameIdLeakage": 0,
              "cohort": {s: {"games": len({r["gameId"] for r in rr}), "pairs": len(rr)}
                         for s, rr in rows.items()},
              "candidateBundle": bundle, "candidateBundleSha256": hashlib.sha256(json_bytes(bundle)).hexdigest(),
              "dataset": provenance, "validation": result,
              "trainHistoryCounts": dict(sorted(Counter(r["historyStatus"] for r in train_history).items())),
              "trainCellCounts": {k: len(v) for k, v in bundle["historyCells"].items()},
              "primary": "validation_game_macro_crps_B3_minus_B2",
              "candidateSelection": "needs_policy_decision" if mode == "full" else "not_eligible",
              "baselineDefinitionsSha256": BASELINE_DEFS_SHA, "sharedNotebookSha256": SHARED_NOTEBOOK_SHA,
              "sourceNotebookSha256": watched[SOURCE_NOTEBOOK],
              "environment": {"python": platform.python_version(), "numpy": np.__version__},
              "inputUnchanged": True,
              "limits": ["teacher_forced_one_step_not_rollout", "recorded_attempt_interval_not_keypress",
                         "no_action_hold_state_or_personalization", "legacy_ruleset_not_current_runtime",
                         "validation_exploration_after_observed_test"]}
    require(all(sha256_file(path) == digest for path, digest in watched.items()), "INPUT_CHANGED")
    json_bytes(report)
    if destination is not None:
        publish_report(report, destination)
    return report


In [ ]:
def show_history(report):
    """저장 report만 표시하며 재학습하거나 입력 파일을 읽지 않는다."""
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    result = report["validation"]
    print(f"Mode: {report['runMode']} | synthetic: {report['synthetic']} | Test read: False")
    print("CRPS delta B3 - B2 (negative is lower):", result["deltaB3MinusB2Ms"])
    display(pd.DataFrame([{k: v for k, v in row.items() if k not in ("predictedCdf", "routes")}
                          for row in result["metrics"]]).set_index("model"))
    display(pd.DataFrame(result["perGame"]).pivot(index="gameId", columns="model", values="crpsMs"))
    display(pd.DataFrame(result["historySubsets"]))
    display(pd.DataFrame([{"context": k, "trainPairs": n}
                          for k, n in report["trainCellCounts"].items()]))
    display(pd.DataFrame(result["perAction"]))
    print("Bucket cuts (ms):", report["candidateBundle"]["bucketCutsMs"])
    print("History coverage:", result["historyCounts"])
    print("Routing:", {row["model"]: row["routes"] for row in result["metrics"]})
    print("Game improvement / tie / worsening:",
          result["improvedGames"], result["tiedGames"], result["worsenedGames"])

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    cdf = result["observedCdf"]
    axes[0].step([0, *cdf["ms"]], [0, *cdf["probability"]], where="post", label="Observed")
    for row in result["metrics"]:
        cdf = row["predictedCdf"]
        axes[0].step([0, *cdf["ms"]], [0, *cdf["probability"]], where="post", label=row["model"])
    axes[0].set(xlabel="Recorded interval (ms)", ylabel="Cumulative probability",
                title=f"{report['runMode']} validation CDF")
    axes[0].legend()
    deltas = result["gameComparison"]
    axes[1].bar(range(len(deltas)), [d["deltaB3MinusB2Ms"] for d in deltas])
    axes[1].axhline(0, color="black", linewidth=.8)
    axes[1].set(xlabel="Game index (Unicode gameId order)", ylabel="CRPS delta (ms)",
                title="B3 - B2; negative is lower")
    names = ["Observed", "B2", "B3"]
    zeros = [result["observed"]["zeroProbability"],
             *[r["predicted"]["zeroProbability"] for r in result["metrics"]]]
    axes[2].bar(names, [p * 100 for p in zeros])
    axes[2].set(ylabel="Zero interval (%)", title="Whole validation; no clipping")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


# 3. Smoke — 가상 예제로 먼저 확인
기본 모두 실행은 아래 작은 가상 예제만 처리합니다. 실제 Test·원본 로그·dataset을 열지 않습니다.
가상 결과로 모델 성능을 판단하지 않습니다. 반복 실행은 메모리 결과라 파일 충돌이 없습니다.


In [ ]:
SMOKE_OUTPUT = os.environ.get("HUMAN_TIMING_HISTORY_SMOKE_REPORT") or None


In [ ]:
SMOKE_REPORT = run_history("smoke", SMOKE_OUTPUT, synthetic=True)
print(SMOKE_REPORT["status"])


In [ ]:
show_history(SMOKE_REPORT)


# 4. Full — 기존 Train/Validation 전체 비교
직접 실행하기로 했을 때만 아래 RUN_FULL을 True로 바꿉니다. 설정 → 실행 → 5번 결과 셀 순서입니다.
Python 테스트 파일을 먼저 실행할 필요는 없습니다. 기존 report가 있으면 FULL_OUTPUT 이름을 바꿉니다.
Train 149게임/41,151 pair, Validation 31게임/10,180 pair를 사용합니다. 기존 Test는 열지 않습니다.
과거 실측을 관측한 한 단계 예측이며 자체 실행·체감·production 평가는 아닙니다.


In [ ]:
RUN_FULL = False
FULL_OUTPUT = os.environ.get("HUMAN_TIMING_HISTORY_FULL_REPORT",
    str(ROOT / "data/ml-experiments/human-timing-history-baseline/full-report-run1.json"))


In [ ]:
FULL_REPORT = None
if RUN_FULL:
    FULL_REPORT = run_history("full", FULL_OUTPUT)
    print(FULL_REPORT["status"])
else:
    print("Full skipped: RUN_FULL=False. Train/Validation/Test files were not opened.")


# 5. Full 결과 읽기
B3 - B2가 음수면 이 Validation에서 과거 간격 문맥이 도움을 준 것입니다. 양수면 악화입니다.
전체 game-macro가 primary이며 history-available subset을 대신 쓰지 않습니다.
Game별 악화·작은 Train cell·fallback을 함께 봅니다. 개선되지 않아도 정상적인 실험 결론입니다.


In [ ]:
if FULL_REPORT is not None:
    show_history(FULL_REPORT)
else:
    print("Full result is not available. Smoke is synthetic, not Human validation performance.")


# 6. 한계와 다음 gate
이전 간격 하나만 추가한 탐색적 경험 분포입니다. 연속 행동 순서·개인 습관·Hold·배치를 학습하지 않았습니다.
Validation의 실제 과거 간격을 제공했으므로 모델 스스로 움직일 때의 리듬은 아직 모릅니다.
이미 본 Test로 후보를 다시 고르지 않습니다. Full 결과를 분석한 뒤 다음 가설이나 새로운 독립 cohort를 별도로 결정합니다.
기존 0ms 원인 조사는 재개하지 않고 원본과 source Notebook의 사용자 실행 출력을 보존합니다.
